In [1]:
# Phase 1A subset-aware deep learning 실험 준비 셀
# 목적:
# - phase_1a_experiment_grid.csv를 읽는다.
# - deep_learning_sequences_by_subset 경로에서 subset/window/split별 tensor를 불러온다.
# - 이후 학습 셀에서 공통으로 사용할 Dataset/DataLoader를 준비한다.

from pathlib import Path
import random
import json

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader


PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")

SEQUENCE_ROOT = PROJECT_ROOT / "data" / "processed" / "deep_learning_sequences_by_subset"
EXPERIMENT_ROOT = PROJECT_ROOT / "data" / "processed" / "deep_learning_subset_experiments"
GRID_PATH = EXPERIMENT_ROOT / "phase_1a_experiment_grid.csv"

OUTPUT_ROOT = EXPERIMENT_ROOT / "phase_1a_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SEED = 42
BATCH_SIZE = 32
NUM_EPOCHS = 80
PATIENCE = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SEQUENCE_ROOT exists:", SEQUENCE_ROOT.exists())
print("GRID_PATH exists:", GRID_PATH.exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("DEVICE:", DEVICE)


experiment_grid = pd.read_csv(GRID_PATH, encoding="utf-8-sig")

print("phase_1a experiments:", len(experiment_grid))
display(experiment_grid.head())
display(
    experiment_grid
    .groupby(["subset_name", "window"], as_index=False)
    .agg(experiments=("model_family", "count"))
)


class SleepTensorDataset(Dataset):
    # 하나의 split tensor를 PyTorch Dataset으로 감싸는 클래스
    def __init__(self, arrays, input_key):
        self.X = torch.tensor(arrays[input_key], dtype=torch.float32)
        self.y = torch.tensor(arrays["y"], dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def model_input_key(model_family):
    # 모델 종류에 따라 사용할 입력 tensor를 결정한다.
    if model_family == "mlp_current_day":
        return "X_mlp_current_day"

    if model_family == "mlp_flattened_window":
        return "X_mlp_flattened"

    return "X_sequence"


def load_split_arrays(subset_name, window, split):
    # subset/window/split에 해당하는 npz tensor를 불러온다.
    path = SEQUENCE_ROOT / subset_name / f"window_{window}" / f"{split}.npz"

    if not path.exists():
        raise FileNotFoundError(path)

    return np.load(path, allow_pickle=True)


def load_experiment_data(subset_name, window, model_family, batch_size=BATCH_SIZE):
    # 하나의 실험 조합에 필요한 train/validation/test DataLoader를 생성한다.
    input_key = model_input_key(model_family)

    train_arrays = load_split_arrays(subset_name, window, "train")
    validation_arrays = load_split_arrays(subset_name, window, "validation")
    test_arrays = load_split_arrays(subset_name, window, "test")

    train_dataset = SleepTensorDataset(train_arrays, input_key)
    validation_dataset = SleepTensorDataset(validation_arrays, input_key)
    test_dataset = SleepTensorDataset(test_arrays, input_key)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )

    input_shape = tuple(train_dataset.X.shape[1:])
    positive_rate = float(train_dataset.y.mean().item())

    return {
        "input_key": input_key,
        "input_shape": input_shape,
        "train_loader": train_loader,
        "validation_loader": validation_loader,
        "test_loader": test_loader,
        "train_arrays": train_arrays,
        "validation_arrays": validation_arrays,
        "test_arrays": test_arrays,
        "positive_rate": positive_rate,
    }


# 대표 조합 하나를 불러와서 shape만 확인한다.
sample_row = experiment_grid.iloc[0]

sample_data = load_experiment_data(
    subset_name=sample_row["subset_name"],
    window=int(sample_row["window"]),
    model_family=sample_row["model_family"],
)

print("sample experiment:")
print(sample_row.to_dict())
print("input_key:", sample_data["input_key"])
print("input_shape:", sample_data["input_shape"])
print("train positive_rate:", sample_data["positive_rate"])

batch_X, batch_y = next(iter(sample_data["train_loader"]))
print("batch_X shape:", batch_X.shape)
print("batch_y shape:", batch_y.shape)

PROJECT_ROOT: c:\workSpace\DeepLearnin_sleep
SEQUENCE_ROOT exists: True
GRID_PATH exists: True
OUTPUT_ROOT: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_subset_experiments\phase_1a_outputs
DEVICE: cpu
phase_1a experiments: 36


,experiment_id,phase,subset_name,window,model_family,run,selection_metric,test_policy
0,phase1a_000,phase_1_priority,daytime_only,7,bilstm,True,validation_balanced_accuracy,validation ranking 이후 확인
1,phase1a_001,phase_1_priority,daytime_only,7,cnn_1d,True,validation_balanced_accuracy,validation ranking 이후 확인
2,phase1a_002,phase_1_priority,daytime_only,7,gru,True,validation_balanced_accuracy,validation ranking 이후 확인
3,phase1a_003,phase_1_priority,daytime_only,7,mlp_current_day,True,validation_balanced_accuracy,validation ranking 이후 확인
4,phase1a_004,phase_1_priority,daytime_only,14,bilstm,True,validation_balanced_accuracy,validation ranking 이후 확인


,subset_name,window,experiments
0,daytime_only,7,4
1,daytime_only,14,4
2,daytime_only,30,4
3,full_current,7,4
4,full_current,14,4
5,full_current,30,4
6,no_high_sleep_session,7,4
7,no_high_sleep_session,14,4
8,no_high_sleep_session,30,4


sample experiment:
{'experiment_id': 'phase1a_000', 'phase': 'phase_1_priority', 'subset_name': 'daytime_only', 'window': 7, 'model_family': 'bilstm', 'run': True, 'selection_metric': 'validation_balanced_accuracy', 'test_policy': 'validation ranking 이후 확인'}
input_key: X_sequence
input_shape: (7, 19)
train positive_rate: 0.4034458100795746
batch_X shape: torch.Size([32, 7, 19])
batch_y shape: torch.Size([32])


In [2]:
# Phase 1A 모델 정의 셀
# 목적:
# - Phase 1A에서 사용할 모델 4개를 정의한다.
# - mlp_current_day는 현재 날짜 feature만 사용한다.
# - GRU/BiLSTM/1D CNN은 sequence tensor를 사용한다.

class MLPCurrentDay(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.30):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class GRUClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1, dropout=0.20):
        super().__init__()

        effective_dropout = dropout if num_layers > 1 else 0.0

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=effective_dropout,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        output, hidden = self.gru(x)
        last_hidden = hidden[-1]
        return self.head(last_hidden).squeeze(-1)


class BiLSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=48, num_layers=1, dropout=0.20):
        super().__init__()

        effective_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=effective_dropout,
            bidirectional=True,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, 1),
        )

    def forward(self, x):
        output, (hidden, cell) = self.lstm(x)

        forward_last = hidden[-2]
        backward_last = hidden[-1]
        combined = torch.cat([forward_last, backward_last], dim=1)

        return self.head(combined).squeeze(-1)


class CNN1DClassifier(nn.Module):
    def __init__(self, input_dim, channels=64, dropout=0.25):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels),
            nn.ReLU(),
        )

        self.pool = nn.AdaptiveMaxPool1d(1)

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(channels, 1),
        )

    def forward(self, x):
        # 입력: batch x time x features
        # Conv1d 입력: batch x features x time
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = self.pool(x).squeeze(-1)
        return self.head(x).squeeze(-1)


def build_model(model_family, input_shape):
    # input_shape:
    # - MLP: (features,)
    # - sequence model: (window, features)

    if model_family == "mlp_current_day":
        input_dim = input_shape[0]
        return MLPCurrentDay(input_dim=input_dim)

    if model_family == "gru":
        input_dim = input_shape[-1]
        return GRUClassifier(input_dim=input_dim)

    if model_family == "bilstm":
        input_dim = input_shape[-1]
        return BiLSTMClassifier(input_dim=input_dim)

    if model_family == "cnn_1d":
        input_dim = input_shape[-1]
        return CNN1DClassifier(input_dim=input_dim)

    raise ValueError(f"지원하지 않는 model_family입니다: {model_family}")


# 모델 생성 smoke test
for model_family in ["mlp_current_day", "gru", "bilstm", "cnn_1d"]:
    sample_data = load_experiment_data(
        subset_name="daytime_only",
        window=7,
        model_family=model_family,
    )

    model = build_model(model_family, sample_data["input_shape"]).to(DEVICE)
    batch_X, batch_y = next(iter(sample_data["train_loader"]))

    with torch.no_grad():
        logits = model(batch_X.to(DEVICE))

    print(model_family)
    print("  input_shape:", sample_data["input_shape"])
    print("  batch_X:", tuple(batch_X.shape))
    print("  logits:", tuple(logits.shape))

mlp_current_day
  input_shape: (19,)
  batch_X: (32, 19)
  logits: (32,)
gru
  input_shape: (7, 19)
  batch_X: (32, 7, 19)
  logits: (32,)
bilstm
  input_shape: (7, 19)
  batch_X: (32, 7, 19)
  logits: (32,)
cnn_1d
  input_shape: (7, 19)
  batch_X: (32, 7, 19)
  logits: (32,)


In [3]:
# Phase 1A 학습/평가 함수 정의 셀
# 목적:
# - 하나의 실험을 학습하고 validation/test metric을 계산하는 공통 함수를 정의한다.
# - validation balanced accuracy 기준으로 early stopping을 적용한다.
# - test metric은 계산 가능하지만, 해석은 validation ranking 이후에만 한다.

from copy import deepcopy
from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)


def make_pos_weight(train_arrays):
    # BCEWithLogitsLoss의 class imbalance 보정값을 만든다.
    y = train_arrays["y"].astype(int)
    positives = int((y == 1).sum())
    negatives = int((y == 0).sum())

    if positives == 0:
        return torch.tensor(1.0, dtype=torch.float32, device=DEVICE)

    return torch.tensor(negatives / positives, dtype=torch.float32, device=DEVICE)


def predict_logits(model, loader):
    model.eval()

    all_logits = []
    all_y = []

    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(DEVICE)
            logits = model(batch_X)

            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(batch_y.detach().cpu().numpy())

    logits = np.concatenate(all_logits)
    y_true = np.concatenate(all_y).astype(int)

    return logits, y_true


def metrics_from_logits(logits, y_true, threshold=0.5):
    probabilities = 1.0 / (1.0 + np.exp(-logits))
    y_pred = (probabilities >= threshold).astype(int)

    result = {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "positive_prediction_rate": float(y_pred.mean()),
        "target_positive_rate": float(y_true.mean()),
    }

    if len(np.unique(y_true)) == 2:
        result["roc_auc"] = roc_auc_score(y_true, probabilities)
        result["average_precision"] = average_precision_score(y_true, probabilities)
    else:
        result["roc_auc"] = np.nan
        result["average_precision"] = np.nan

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result.update({
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    })

    return result, probabilities, y_pred


def train_one_experiment(row, verbose=True):
    experiment_id = row["experiment_id"]
    subset_name = row["subset_name"]
    window = int(row["window"])
    model_family = row["model_family"]

    data = load_experiment_data(
        subset_name=subset_name,
        window=window,
        model_family=model_family,
    )

    model = build_model(model_family, data["input_shape"]).to(DEVICE)

    pos_weight = make_pos_weight(data["train_arrays"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_state = None
    best_validation_balanced_accuracy = -np.inf
    best_epoch = -1
    patience_counter = 0

    history_rows = []

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        train_losses = []

        for batch_X, batch_y in data["train_loader"]:
            batch_X = batch_X.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            optimizer.step()
            train_losses.append(float(loss.item()))

        validation_logits, validation_y = predict_logits(model, data["validation_loader"])
        validation_metrics, _, _ = metrics_from_logits(validation_logits, validation_y)

        train_loss = float(np.mean(train_losses))

        history_rows.append({
            "experiment_id": experiment_id,
            "subset_name": subset_name,
            "window": window,
            "model_family": model_family,
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_balanced_accuracy": validation_metrics["balanced_accuracy"],
            "validation_roc_auc": validation_metrics["roc_auc"],
            "validation_f1": validation_metrics["f1"],
            "validation_recall": validation_metrics["recall"],
            "validation_precision": validation_metrics["precision"],
        })

        current_score = validation_metrics["balanced_accuracy"]

        if current_score > best_validation_balanced_accuracy:
            best_validation_balanced_accuracy = current_score
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(
                f"{experiment_id} epoch={epoch:03d} "
                f"loss={train_loss:.4f} "
                f"val_bal_acc={current_score:.4f}"
            )

        if patience_counter >= PATIENCE:
            break

    model.load_state_dict(best_state)

    metric_rows = []
    prediction_rows = []

    for split_name, loader in [
        ("validation", data["validation_loader"]),
        ("test", data["test_loader"]),
    ]:
        logits, y_true = predict_logits(model, loader)
        metrics, probabilities, y_pred = metrics_from_logits(logits, y_true)

        metric_row = {
            "experiment_id": experiment_id,
            "subset_name": subset_name,
            "window": window,
            "model_family": model_family,
            "split": split_name,
            "best_epoch": best_epoch,
            "input_key": data["input_key"],
            "input_shape": str(data["input_shape"]),
        }
        metric_row.update(metrics)
        metric_rows.append(metric_row)

        arrays = data[f"{split_name}_arrays"]

        split_predictions = pd.DataFrame({
            "experiment_id": experiment_id,
            "subset_name": subset_name,
            "window": window,
            "model_family": model_family,
            "split": split_name,
            "participant_object_id": arrays["participant_object_id"],
            "window_start_date": arrays["window_start_date"],
            "window_end_date": arrays["window_end_date"],
            "y_true": y_true,
            "probability": probabilities,
            "y_pred": y_pred,
        })

        prediction_rows.append(split_predictions)

    metrics_df = pd.DataFrame(metric_rows)
    history_df = pd.DataFrame(history_rows)
    predictions_df = pd.concat(prediction_rows, ignore_index=True)

    return metrics_df, history_df, predictions_df

In [4]:
# Phase 1A smoke training 셀
# 목적:
# - 전체 36개를 돌리기 전에 실험 1개만 학습한다.
# - 학습 루프, early stopping, validation/test metric, prediction 생성이 정상인지 확인한다.
# - 빠른 확인을 위해 임시로 epoch/patience를 작게 줄인다.

ORIGINAL_NUM_EPOCHS = NUM_EPOCHS
ORIGINAL_PATIENCE = PATIENCE

NUM_EPOCHS = 5
PATIENCE = 3

smoke_row = experiment_grid.iloc[0].copy()

print("smoke experiment:")
print(smoke_row.to_dict())

smoke_metrics, smoke_history, smoke_predictions = train_one_experiment(
    smoke_row,
    verbose=True,
)

NUM_EPOCHS = ORIGINAL_NUM_EPOCHS
PATIENCE = ORIGINAL_PATIENCE

print("\nsmoke_metrics:")
display(smoke_metrics)

print("\nsmoke_history tail:")
display(smoke_history.tail())

print("\nsmoke_predictions head:")
display(smoke_predictions.head())

print("metrics shape:", smoke_metrics.shape)
print("history shape:", smoke_history.shape)
print("predictions shape:", smoke_predictions.shape)

smoke experiment:
{'experiment_id': 'phase1a_000', 'phase': 'phase_1_priority', 'subset_name': 'daytime_only', 'window': 7, 'model_family': 'bilstm', 'run': True, 'selection_metric': 'validation_balanced_accuracy', 'test_policy': 'validation ranking 이후 확인'}
phase1a_000 epoch=001 loss=0.7442 val_bal_acc=0.7435

smoke_metrics:


,experiment_id,subset_name,window,model_family,split,best_epoch,input_key,input_shape,threshold,accuracy,...,precision,recall,positive_prediction_rate,target_positive_rate,roc_auc,average_precision,tn,fp,fn,tp
0,phase1a_000,daytime_only,7,bilstm,validation,4,X_sequence,"(7, 19)",0.5,0.799257,...,0.720339,0.801887,0.438662,0.394052,0.850156,0.757107,130,33,21,85
1,phase1a_000,daytime_only,7,bilstm,test,4,X_sequence,"(7, 19)",0.5,0.770701,...,0.700000,0.795455,0.477707,0.420382,0.864344,0.795215,137,45,27,105



smoke_history tail:


,experiment_id,subset_name,window,model_family,epoch,train_loss,validation_balanced_accuracy,validation_roc_auc,validation_f1,validation_recall,validation_precision
0,phase1a_000,daytime_only,7,bilstm,1,0.744179,0.743547,0.812594,0.705882,0.849057,0.604027
1,phase1a_000,daytime_only,7,bilstm,2,0.630451,0.757032,0.836266,0.704762,0.698113,0.711538
2,phase1a_000,daytime_only,7,bilstm,3,0.570846,0.758682,0.851256,0.707547,0.707547,0.707547
3,phase1a_000,daytime_only,7,bilstm,4,0.513275,0.799716,0.850156,0.758929,0.801887,0.720339
4,phase1a_000,daytime_only,7,bilstm,5,0.466878,0.788170,0.857738,0.742857,0.735849,0.750000



smoke_predictions head:


,experiment_id,subset_name,window,model_family,split,participant_object_id,window_start_date,window_end_date,y_true,probability,y_pred
0,phase1a_000,daytime_only,7,bilstm,validation,621e2f3967b776a240c654db,2021-05-25,2021-05-31,1,0.966635,1
1,phase1a_000,daytime_only,7,bilstm,validation,621e2f3967b776a240c654db,2021-05-26,2021-06-01,0,0.730412,1
2,phase1a_000,daytime_only,7,bilstm,validation,621e2f3967b776a240c654db,2021-05-27,2021-06-02,0,0.086695,0
3,phase1a_000,daytime_only,7,bilstm,validation,621e2f3967b776a240c654db,2021-05-28,2021-06-03,1,0.642099,1
4,phase1a_000,daytime_only,7,bilstm,validation,621e2f3967b776a240c654db,2021-05-29,2021-06-04,0,0.691969,1


metrics shape: (2, 22)
history shape: (5, 11)
predictions shape: (583, 11)


In [6]:
# Phase 1A 전체 학습 실행 셀
# 목적:
# - phase_1a_experiment_grid.csv에 정의된 36개 실험을 실행한다.
# - 각 실험마다 metrics/history/predictions를 누적 저장한다.
# - 중간에 멈춰도 이미 완료된 experiment_id는 건너뛸 수 있게 한다.
# - test metric은 저장만 하고, 최종 해석은 validation ranking 이후에 한다.

METRICS_PATH = OUTPUT_ROOT / "phase_1a_model_metrics.csv"
HISTORY_PATH = OUTPUT_ROOT / "phase_1a_training_history.csv"
PREDICTIONS_PATH = OUTPUT_ROOT / "phase_1a_model_predictions.csv"

RUN_PHASE_1A_FULL = True

print("METRICS_PATH:", METRICS_PATH)
print("HISTORY_PATH:", HISTORY_PATH)
print("PREDICTIONS_PATH:", PREDICTIONS_PATH)
print("RUN_PHASE_1A_FULL:", RUN_PHASE_1A_FULL)

if RUN_PHASE_1A_FULL:
    completed_experiment_ids = set()

    if METRICS_PATH.exists():
        existing_metrics = pd.read_csv(METRICS_PATH, encoding="utf-8-sig")
        completed_experiment_ids = set(existing_metrics["experiment_id"].unique())
        print("completed experiments:", len(completed_experiment_ids))

    metrics_outputs = []
    history_outputs = []
    prediction_outputs = []

    for idx, row in experiment_grid.iterrows():
        experiment_id = row["experiment_id"]

        if experiment_id in completed_experiment_ids:
            print(f"[SKIP] {experiment_id} already completed")
            continue

        print("\n" + "=" * 100)
        print(
            f"[RUN] {idx + 1}/{len(experiment_grid)} "
            f"{experiment_id} | {row['subset_name']} | "
            f"window={row['window']} | model={row['model_family']}"
        )

        metrics_df, history_df, predictions_df = train_one_experiment(
            row,
            verbose=True,
        )

        metrics_outputs.append(metrics_df)
        history_outputs.append(history_df)
        prediction_outputs.append(predictions_df)

        if metrics_outputs:
            new_metrics = pd.concat(metrics_outputs, ignore_index=True)

            if METRICS_PATH.exists():
                old_metrics = pd.read_csv(METRICS_PATH, encoding="utf-8-sig")
                save_metrics = pd.concat([old_metrics, new_metrics], ignore_index=True)
                save_metrics = save_metrics.drop_duplicates(
                    subset=["experiment_id", "split"],
                    keep="last",
                )
            else:
                save_metrics = new_metrics

            save_metrics.to_csv(METRICS_PATH, index=False, encoding="utf-8-sig")

        if history_outputs:
            new_history = pd.concat(history_outputs, ignore_index=True)

            if HISTORY_PATH.exists():
                old_history = pd.read_csv(HISTORY_PATH, encoding="utf-8-sig")
                save_history = pd.concat([old_history, new_history], ignore_index=True)
                save_history = save_history.drop_duplicates(
                    subset=["experiment_id", "epoch"],
                    keep="last",
                )
            else:
                save_history = new_history

            save_history.to_csv(HISTORY_PATH, index=False, encoding="utf-8-sig")

        if prediction_outputs:
            new_predictions = pd.concat(prediction_outputs, ignore_index=True)

            if PREDICTIONS_PATH.exists():
                old_predictions = pd.read_csv(PREDICTIONS_PATH, encoding="utf-8-sig")
                save_predictions = pd.concat([old_predictions, new_predictions], ignore_index=True)
                save_predictions = save_predictions.drop_duplicates(
                    subset=[
                        "experiment_id",
                        "split",
                        "participant_object_id",
                        "window_start_date",
                        "window_end_date",
                    ],
                    keep="last",
                )
            else:
                save_predictions = new_predictions

            save_predictions.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

        print(f"[SAVED] {experiment_id}")

    print("\nPhase 1A 학습 완료")
else:
    print("RUN_PHASE_1A_FULL이 False입니다.")
    print("전체 36개 실험을 실행하려면 RUN_PHASE_1A_FULL = True로 바꾸고 이 셀을 다시 실행하세요.")

METRICS_PATH: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_subset_experiments\phase_1a_outputs\phase_1a_model_metrics.csv
HISTORY_PATH: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_subset_experiments\phase_1a_outputs\phase_1a_training_history.csv
PREDICTIONS_PATH: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_subset_experiments\phase_1a_outputs\phase_1a_model_predictions.csv
RUN_PHASE_1A_FULL: True

[RUN] 1/36 phase1a_000 | daytime_only | window=7 | model=bilstm
phase1a_000 epoch=001 loss=0.7388 val_bal_acc=0.7332
phase1a_000 epoch=010 loss=0.3572 val_bal_acc=0.8339
phase1a_000 epoch=020 loss=0.2795 val_bal_acc=0.7752
[SAVED] phase1a_000

[RUN] 2/36 phase1a_001 | daytime_only | window=7 | model=cnn_1d
phase1a_001 epoch=001 loss=0.8110 val_bal_acc=0.5860
phase1a_001 epoch=010 loss=0.4344 val_bal_acc=0.7813
phase1a_001 epoch=020 loss=0.3186 val_bal_acc=0.7787
phase1a_001 epoch=030 loss=0.2752 val_bal_acc=0.8412
phase1a_001 epoch=040 loss=0.2626

In [7]:
# Phase 1A 결과 확인 셀
# 목적:
# - validation balanced accuracy 기준으로 모델/subset/window를 정렬한다.
# - validation에서 선택된 후보의 test 성능을 나중에 붙여서 확인한다.
# - full_current, no_high_sleep_session, daytime_only의 일반화 차이를 본다.

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"c:\workSpace\DeepLearnin_sleep")
OUTPUT_ROOT = PROJECT_ROOT / "data" / "processed" / "deep_learning_subset_experiments" / "phase_1a_outputs"

METRICS_PATH = OUTPUT_ROOT / "phase_1a_model_metrics.csv"
HISTORY_PATH = OUTPUT_ROOT / "phase_1a_training_history.csv"
PREDICTIONS_PATH = OUTPUT_ROOT / "phase_1a_model_predictions.csv"

metrics = pd.read_csv(METRICS_PATH, encoding="utf-8-sig")
history = pd.read_csv(HISTORY_PATH, encoding="utf-8-sig")
predictions = pd.read_csv(PREDICTIONS_PATH, encoding="utf-8-sig")

print("metrics shape:", metrics.shape)
print("history shape:", history.shape)
print("predictions shape:", predictions.shape)
print("experiments in metrics:", metrics["experiment_id"].nunique())
print(metrics["split"].value_counts())

validation_metrics = metrics[metrics["split"] == "validation"].copy()
test_metrics = metrics[metrics["split"] == "test"].copy()

validation_rank = validation_metrics.sort_values(
    ["balanced_accuracy", "roc_auc", "average_precision"],
    ascending=False,
).reset_index(drop=True)

display(validation_rank.head(20))

paired_results = validation_metrics.merge(
    test_metrics,
    on=["experiment_id", "subset_name", "window", "model_family"],
    suffixes=("_validation", "_test"),
)

paired_results["balanced_accuracy_gap"] = (
    paired_results["balanced_accuracy_validation"]
    - paired_results["balanced_accuracy_test"]
)

paired_results["roc_auc_gap"] = (
    paired_results["roc_auc_validation"]
    - paired_results["roc_auc_test"]
)

paired_rank = paired_results.sort_values(
    ["balanced_accuracy_validation", "roc_auc_validation", "average_precision_validation"],
    ascending=False,
).reset_index(drop=True)

display(
    paired_rank[
        [
            "experiment_id",
            "subset_name",
            "window",
            "model_family",
            "best_epoch_validation",
            "balanced_accuracy_validation",
            "roc_auc_validation",
            "average_precision_validation",
            "f1_validation",
            "balanced_accuracy_test",
            "roc_auc_test",
            "average_precision_test",
            "f1_test",
            "balanced_accuracy_gap",
            "roc_auc_gap",
        ]
    ].head(20)
)

subset_summary = (
    paired_results
    .groupby("subset_name", as_index=False)
    .agg(
        experiments=("experiment_id", "nunique"),
        mean_val_bal_acc=("balanced_accuracy_validation", "mean"),
        max_val_bal_acc=("balanced_accuracy_validation", "max"),
        mean_test_bal_acc=("balanced_accuracy_test", "mean"),
        max_test_bal_acc=("balanced_accuracy_test", "max"),
        mean_bal_acc_gap=("balanced_accuracy_gap", "mean"),
        median_bal_acc_gap=("balanced_accuracy_gap", "median"),
    )
    .sort_values("max_val_bal_acc", ascending=False)
)

display(subset_summary)

model_summary = (
    paired_results
    .groupby("model_family", as_index=False)
    .agg(
        experiments=("experiment_id", "nunique"),
        mean_val_bal_acc=("balanced_accuracy_validation", "mean"),
        max_val_bal_acc=("balanced_accuracy_validation", "max"),
        mean_test_bal_acc=("balanced_accuracy_test", "mean"),
        max_test_bal_acc=("balanced_accuracy_test", "max"),
        mean_bal_acc_gap=("balanced_accuracy_gap", "mean"),
    )
    .sort_values("max_val_bal_acc", ascending=False)
)

display(model_summary)

window_summary = (
    paired_results
    .groupby("window", as_index=False)
    .agg(
        experiments=("experiment_id", "nunique"),
        mean_val_bal_acc=("balanced_accuracy_validation", "mean"),
        max_val_bal_acc=("balanced_accuracy_validation", "max"),
        mean_test_bal_acc=("balanced_accuracy_test", "mean"),
        max_test_bal_acc=("balanced_accuracy_test", "max"),
        mean_bal_acc_gap=("balanced_accuracy_gap", "mean"),
    )
    .sort_values("window")
)

display(window_summary)

metrics shape: (72, 22)
history shape: (1053, 11)
predictions shape: (13248, 11)
experiments in metrics: 36
split
validation    36
test          36
Name: count, dtype: int64


,experiment_id,subset_name,window,model_family,split,best_epoch,input_key,input_shape,threshold,accuracy,...,precision,recall,positive_prediction_rate,target_positive_rate,roc_auc,average_precision,tn,fp,fn,tp
0,phase1a_022,full_current,30,gru,validation,23,X_sequence,"(30, 197)",0.5,0.890909,...,0.684211,1.000000,0.345455,0.236364,0.950549,0.833623,36,6,0,13
1,phase1a_035,no_high_sleep_session,30,mlp_current_day,validation,16,X_mlp_current_day,"(127,)",0.5,0.872727,...,0.650000,1.000000,0.363636,0.236364,0.915751,0.721537,35,7,0,13
2,phase1a_008,daytime_only,30,bilstm,validation,6,X_sequence,"(30, 19)",0.5,0.909091,...,0.750000,0.923077,0.290909,0.236364,0.939560,0.753736,38,4,1,12
3,phase1a_010,daytime_only,30,gru,validation,5,X_sequence,"(30, 19)",0.5,0.909091,...,0.750000,0.923077,0.290909,0.236364,0.937729,0.789460,38,4,1,12
4,phase1a_011,daytime_only,30,mlp_current_day,validation,4,X_mlp_current_day,"(19,)",0.5,0.854545,...,0.619048,1.000000,0.381818,0.236364,0.979853,0.938001,34,8,0,13
5,phase1a_032,no_high_sleep_session,30,bilstm,validation,11,X_sequence,"(30, 127)",0.5,0.854545,...,0.619048,1.000000,0.381818,0.236364,0.926740,0.709782,34,8,0,13
6,phase1a_034,no_high_sleep_session,30,gru,validation,15,X_sequence,"(30, 127)",0.5,0.909091,...,0.785714,0.846154,0.254545,0.236364,0.930403,0.740096,39,3,2,11
7,phase1a_003,daytime_only,7,mlp_current_day,validation,38,X_mlp_current_day,"(19,)",0.5,0.873606,...,0.795082,0.915094,0.453532,0.394052,0.908844,0.791541,138,25,9,97
8,phase1a_023,full_current,30,mlp_current_day,validation,16,X_mlp_current_day,"(197,)",0.5,0.854545,...,0.631579,0.923077,0.345455,0.236364,0.851648,0.628546,35,7,1,12
9,phase1a_007,daytime_only,14,mlp_current_day,validation,14,X_mlp_current_day,"(19,)",0.5,0.876712,...,0.789474,0.882353,0.390411,0.349315,0.906295,0.813508,83,12,6,45


,experiment_id,subset_name,window,model_family,best_epoch_validation,balanced_accuracy_validation,roc_auc_validation,average_precision_validation,f1_validation,balanced_accuracy_test,roc_auc_test,average_precision_test,f1_test,balanced_accuracy_gap,roc_auc_gap
0,phase1a_022,full_current,30,gru,23,0.928571,0.950549,0.833623,0.812500,0.479513,0.566630,0.604560,0.333333,0.449059,0.383920
1,phase1a_035,no_high_sleep_session,30,mlp_current_day,16,0.916667,0.915751,0.721537,0.787879,0.808047,0.838686,0.808309,0.838710,0.108619,0.077065
2,phase1a_008,daytime_only,30,bilstm,6,0.913919,0.939560,0.753736,0.827586,0.638427,0.816537,0.877079,0.775510,0.275492,0.123023
3,phase1a_010,daytime_only,30,gru,5,0.913919,0.937729,0.789460,0.827586,0.777962,0.849022,0.882772,0.844444,0.135957,0.088707
4,phase1a_011,daytime_only,30,mlp_current_day,4,0.904762,0.979853,0.938001,0.764706,0.654854,0.847545,0.869991,0.805195,0.249908,0.132308
5,phase1a_032,no_high_sleep_session,30,bilstm,11,0.904762,0.926740,0.709782,0.764706,0.569029,0.551495,0.639763,0.628099,0.335733,0.375245
6,phase1a_034,no_high_sleep_session,30,gru,15,0.887363,0.930403,0.740096,0.814815,0.558324,0.616464,0.679834,0.463158,0.329039,0.313939
7,phase1a_003,daytime_only,7,mlp_current_day,38,0.880860,0.908844,0.791541,0.850877,0.851856,0.902264,0.840232,0.829268,0.029004,0.006579
8,phase1a_023,full_current,30,mlp_current_day,16,0.878205,0.851648,0.628546,0.750000,0.591178,0.660391,0.759387,0.587156,0.287028,0.191257
9,phase1a_007,daytime_only,14,mlp_current_day,14,0.878019,0.906295,0.813508,0.833333,0.839638,0.877199,0.833664,0.830918,0.038381,0.029096


,subset_name,experiments,mean_val_bal_acc,max_val_bal_acc,mean_test_bal_acc,max_test_bal_acc,mean_bal_acc_gap,median_bal_acc_gap
1,full_current,12,0.824005,0.928571,0.604166,0.727980,0.219838,0.166020
2,no_high_sleep_session,12,0.841987,0.916667,0.676401,0.808047,0.165585,0.125068
0,daytime_only,12,0.864202,0.913919,0.768645,0.851856,0.095557,0.038709


,model_family,experiments,mean_val_bal_acc,max_val_bal_acc,mean_test_bal_acc,max_test_bal_acc,mean_bal_acc_gap
2,gru,9,0.857940,0.928571,0.679909,0.842865,0.178031
3,mlp_current_day,9,0.870761,0.916667,0.752490,0.851856,0.118271
0,bilstm,9,0.826982,0.913919,0.659005,0.831809,0.167977
1,cnn_1d,9,0.817907,0.872844,0.640879,0.804321,0.177028


,window,experiments,mean_val_bal_acc,max_val_bal_acc,mean_test_bal_acc,max_test_bal_acc,mean_bal_acc_gap
0,7,12,0.817246,0.880860,0.750069,0.851856,0.067177
1,14,12,0.836421,0.878019,0.709565,0.839638,0.126856
2,30,12,0.876526,0.928571,0.589578,0.808047,0.286948


In [8]:
# Phase 1A validation threshold tuning
# 목적:
# - 각 experiment_id별로 validation에서 balanced accuracy가 최대인 threshold를 찾는다.
# - 찾은 threshold를 test prediction에 그대로 적용한다.
# - test threshold를 직접 최적화하지 않는다.

from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

predictions = pd.read_csv(PREDICTIONS_PATH, encoding="utf-8-sig")

threshold_grid = np.round(np.arange(0.10, 0.91, 0.01), 2)

def evaluate_with_threshold(y_true, probability, threshold):
    y_pred = (probability >= threshold).astype(int)

    result = {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "positive_prediction_rate": float(y_pred.mean()),
        "target_positive_rate": float(y_true.mean()),
    }

    if len(np.unique(y_true)) == 2:
        result["roc_auc"] = roc_auc_score(y_true, probability)
        result["average_precision"] = average_precision_score(y_true, probability)
    else:
        result["roc_auc"] = np.nan
        result["average_precision"] = np.nan

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result.update({"tn": tn, "fp": fp, "fn": fn, "tp": tp})

    return result


rows = []

for experiment_id, group in predictions.groupby("experiment_id"):
    validation_group = group[group["split"] == "validation"].copy()
    test_group = group[group["split"] == "test"].copy()

    validation_scores = []

    for threshold in threshold_grid:
        metric = evaluate_with_threshold(
            validation_group["y_true"].to_numpy(),
            validation_group["probability"].to_numpy(),
            threshold,
        )
        validation_scores.append(metric)

    validation_threshold_df = pd.DataFrame(validation_scores)

    best_validation_row = validation_threshold_df.sort_values(
        ["balanced_accuracy", "f1", "roc_auc"],
        ascending=False,
    ).iloc[0]

    best_threshold = float(best_validation_row["threshold"])

    for split_name, split_group in [
        ("validation", validation_group),
        ("test", test_group),
    ]:
        metric = evaluate_with_threshold(
            split_group["y_true"].to_numpy(),
            split_group["probability"].to_numpy(),
            best_threshold,
        )

        meta = split_group.iloc[0][
            ["experiment_id", "subset_name", "window", "model_family"]
        ].to_dict()

        rows.append({
            **meta,
            "split": split_name,
            "selected_threshold_from_validation": best_threshold,
            **metric,
        })

threshold_metrics = pd.DataFrame(rows)

validation_threshold_metrics = threshold_metrics[
    threshold_metrics["split"] == "validation"
].copy()

test_threshold_metrics = threshold_metrics[
    threshold_metrics["split"] == "test"
].copy()

paired_threshold_results = validation_threshold_metrics.merge(
    test_threshold_metrics,
    on=["experiment_id", "subset_name", "window", "model_family"],
    suffixes=("_validation", "_test"),
)

paired_threshold_results["balanced_accuracy_gap"] = (
    paired_threshold_results["balanced_accuracy_validation"]
    - paired_threshold_results["balanced_accuracy_test"]
)

paired_threshold_rank = paired_threshold_results.sort_values(
    ["balanced_accuracy_validation", "roc_auc_validation", "average_precision_validation"],
    ascending=False,
).reset_index(drop=True)

display(
    paired_threshold_rank[
        [
            "experiment_id",
            "subset_name",
            "window",
            "model_family",
            "selected_threshold_from_validation_validation",
            "balanced_accuracy_validation",
            "roc_auc_validation",
            "average_precision_validation",
            "f1_validation",
            "balanced_accuracy_test",
            "roc_auc_test",
            "average_precision_test",
            "f1_test",
            "balanced_accuracy_gap",
        ]
    ].head(20)
)

threshold_subset_summary = (
    paired_threshold_results
    .groupby("subset_name", as_index=False)
    .agg(
        experiments=("experiment_id", "nunique"),
        mean_val_bal_acc=("balanced_accuracy_validation", "mean"),
        max_val_bal_acc=("balanced_accuracy_validation", "max"),
        mean_test_bal_acc=("balanced_accuracy_test", "mean"),
        max_test_bal_acc=("balanced_accuracy_test", "max"),
        mean_bal_acc_gap=("balanced_accuracy_gap", "mean"),
        median_bal_acc_gap=("balanced_accuracy_gap", "median"),
    )
    .sort_values("max_val_bal_acc", ascending=False)
)

display(threshold_subset_summary)

threshold_metrics_path = OUTPUT_ROOT / "phase_1a_threshold_tuned_metrics.csv"
threshold_metrics.to_csv(threshold_metrics_path, index=False, encoding="utf-8-sig")

print("saved:", threshold_metrics_path)

,experiment_id,subset_name,window,model_family,selected_threshold_from_validation_validation,balanced_accuracy_validation,roc_auc_validation,average_precision_validation,f1_validation,balanced_accuracy_test,roc_auc_test,average_precision_test,f1_test,balanced_accuracy_gap
0,phase1a_008,daytime_only,30,bilstm,0.43,0.952381,0.939560,0.753736,0.866667,0.642673,0.816537,0.877079,0.786667,0.309708
1,phase1a_011,daytime_only,30,mlp_current_day,0.55,0.940476,0.979853,0.938001,0.838710,0.716685,0.847545,0.869991,0.829932,0.223791
2,phase1a_022,full_current,30,gru,0.46,0.928571,0.950549,0.833623,0.812500,0.483758,0.566630,0.604560,0.365591,0.444814
3,phase1a_035,no_high_sleep_session,30,mlp_current_day,0.51,0.928571,0.915751,0.721537,0.812500,0.800111,0.838686,0.808309,0.829268,0.128461
4,phase1a_010,daytime_only,30,gru,0.49,0.913919,0.937729,0.789460,0.827586,0.785899,0.849022,0.882772,0.852941,0.128021
5,phase1a_032,no_high_sleep_session,30,bilstm,0.44,0.904762,0.926740,0.709782,0.764706,0.523625,0.551495,0.639763,0.651852,0.381137
6,phase1a_034,no_high_sleep_session,30,gru,0.20,0.892857,0.930403,0.740096,0.742857,0.594500,0.616464,0.679834,0.719424,0.298357
7,phase1a_007,daytime_only,14,mlp_current_day,0.43,0.892363,0.906295,0.813508,0.846847,0.829082,0.877199,0.833664,0.824074,0.063282
8,phase1a_003,daytime_only,7,mlp_current_day,0.42,0.891944,0.908844,0.791541,0.862069,0.843989,0.902264,0.840232,0.821549,0.047954
9,phase1a_031,no_high_sleep_session,14,mlp_current_day,0.44,0.878741,0.906295,0.820619,0.838095,0.814567,0.878255,0.838839,0.809302,0.064174


,subset_name,experiments,mean_val_bal_acc,max_val_bal_acc,mean_test_bal_acc,max_test_bal_acc,mean_bal_acc_gap,median_bal_acc_gap
0,daytime_only,12,0.881946,0.952381,0.763507,0.843989,0.118440,0.090810
1,full_current,12,0.831944,0.928571,0.594129,0.729021,0.237816,0.171839
2,no_high_sleep_session,12,0.858180,0.928571,0.667121,0.814567,0.191058,0.164095


saved: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_subset_experiments\phase_1a_outputs\phase_1a_threshold_tuned_metrics.csv


In [9]:
# Phase 1A participant-level stability check
# 목적:
# - validation 기준 상위 후보가 특정 participant에만 잘 맞는지 확인한다.
# - test split에서 participant별 balanced accuracy / positive rate / sample 수를 본다.
# - 최종 후보를 고르기 전에 참가자 단위 안정성을 확인한다.

from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score

TOP_CANDIDATE_IDS = [
    "phase1a_003",  # daytime_only, window 7, mlp_current_day
    "phase1a_002",  # daytime_only, window 7, gru
    "phase1a_007",  # daytime_only, window 14, mlp_current_day
    "phase1a_035",  # no_high_sleep_session, window 30, mlp_current_day
]

threshold_lookup = (
    threshold_metrics[threshold_metrics["split"] == "validation"]
    [["experiment_id", "selected_threshold_from_validation"]]
    .drop_duplicates()
    .set_index("experiment_id")["selected_threshold_from_validation"]
    .to_dict()
)

candidate_prediction_rows = []

for experiment_id in TOP_CANDIDATE_IDS:
    threshold = threshold_lookup[experiment_id]

    candidate_preds = predictions[
        (predictions["experiment_id"] == experiment_id)
        & (predictions["split"] == "test")
    ].copy()

    candidate_preds["selected_threshold"] = threshold
    candidate_preds["y_pred_threshold_tuned"] = (
        candidate_preds["probability"] >= threshold
    ).astype(int)

    candidate_prediction_rows.append(candidate_preds)

candidate_predictions = pd.concat(candidate_prediction_rows, ignore_index=True)

participant_rows = []

for (experiment_id, participant_id), group in candidate_predictions.groupby(
    ["experiment_id", "participant_object_id"]
):
    y_true = group["y_true"].to_numpy()
    y_pred = group["y_pred_threshold_tuned"].to_numpy()

    if len(np.unique(y_true)) == 2:
        bal_acc = balanced_accuracy_score(y_true, y_pred)
    else:
        bal_acc = np.nan

    participant_rows.append({
        "experiment_id": experiment_id,
        "subset_name": group["subset_name"].iloc[0],
        "window": int(group["window"].iloc[0]),
        "model_family": group["model_family"].iloc[0],
        "participant_object_id": participant_id,
        "samples": len(group),
        "target_positive_rate": float(y_true.mean()),
        "prediction_positive_rate": float(y_pred.mean()),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": bal_acc,
        "f1": f1_score(y_true, y_pred, zero_division=0),
    })

participant_metrics = pd.DataFrame(participant_rows)

display(participant_metrics.sort_values(
    ["experiment_id", "balanced_accuracy"],
    ascending=[True, True],
))

participant_stability_summary = (
    participant_metrics
    .groupby(["experiment_id", "subset_name", "window", "model_family"], as_index=False)
    .agg(
        participants=("participant_object_id", "nunique"),
        total_samples=("samples", "sum"),
        mean_participant_accuracy=("accuracy", "mean"),
        min_participant_accuracy=("accuracy", "min"),
        mean_participant_bal_acc=("balanced_accuracy", "mean"),
        min_participant_bal_acc=("balanced_accuracy", "min"),
        participants_with_single_class_target=("balanced_accuracy", lambda s: int(s.isna().sum())),
    )
)

display(participant_stability_summary)

participant_metrics_path = OUTPUT_ROOT / "phase_1a_participant_level_metrics_top_candidates.csv"
participant_metrics.to_csv(participant_metrics_path, index=False, encoding="utf-8-sig")

print("saved:", participant_metrics_path)

,experiment_id,subset_name,window,model_family,participant_object_id,samples,target_positive_rate,prediction_positive_rate,accuracy,balanced_accuracy,f1
1,phase1a_002,daytime_only,7,gru,621e2efa67b776a2409dd1c3,18,0.500000,0.722222,0.777778,0.777778,0.818182
4,phase1a_002,daytime_only,7,gru,621e309b67b776a240b532b0,10,0.100000,0.500000,0.600000,0.777778,0.333333
5,phase1a_002,daytime_only,7,gru,621e324e67b776a2400191cb,56,0.107143,0.500000,0.607143,0.780000,0.352941
7,phase1a_002,daytime_only,7,gru,621e356967b776a24027bd9f,62,0.612903,0.677419,0.838710,0.814693,0.875000
2,phase1a_002,daytime_only,7,gru,621e2f1b67b776a240b3d87c,70,0.585714,0.685714,0.871429,0.849874,0.898876
0,phase1a_002,daytime_only,7,gru,621e2eaf67b776a2406b14ac,62,0.483871,0.516129,0.903226,0.904167,0.903226
6,phase1a_002,daytime_only,7,gru,621e32af67b776a24045b4cf,27,0.222222,0.370370,0.851852,0.904762,0.750000
3,phase1a_002,daytime_only,7,gru,621e309267b776a240ae1cdb,7,0.142857,0.142857,1.000000,1.000000,1.000000
8,phase1a_002,daytime_only,7,gru,621e36c267b776a240ba2756,2,0.000000,0.000000,1.000000,NaN,0.000000
13,phase1a_003,daytime_only,7,mlp_current_day,621e309b67b776a240b532b0,10,0.100000,0.500000,0.600000,0.777778,0.333333


,experiment_id,subset_name,window,model_family,participants,total_samples,mean_participant_accuracy,min_participant_accuracy,mean_participant_bal_acc,min_participant_bal_acc,participants_with_single_class_target
0,phase1a_002,daytime_only,7,gru,9,314,0.827793,0.600000,0.851131,0.777778,1
1,phase1a_003,daytime_only,7,mlp_current_day,9,314,0.847957,0.600000,0.873183,0.777778,1
2,phase1a_007,daytime_only,14,mlp_current_day,5,214,0.816325,0.666667,0.851849,0.803828,0
3,phase1a_035,no_high_sleep_session,30,mlp_current_day,4,106,0.760393,0.600000,0.814151,0.666667,0


saved: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_subset_experiments\phase_1a_outputs\phase_1a_participant_level_metrics_top_candidates.csv


In [10]:
# Phase 1A selected candidate summary 저장
# 목적:
# - validation 기준, test 확인, participant-level 안정성을 합쳐 후보 요약표를 만든다.
# - 최종 선택은 아직 확정하지 않고, Phase 1A 우선 후보로 기록한다.

selected_candidate_ids = [
    "phase1a_003",
    "phase1a_002",
    "phase1a_007",
    "phase1a_035",
]

candidate_summary = paired_threshold_results[
    paired_threshold_results["experiment_id"].isin(selected_candidate_ids)
].copy()

candidate_summary = candidate_summary.merge(
    participant_stability_summary,
    on=["experiment_id", "subset_name", "window", "model_family"],
    how="left",
)

candidate_summary["candidate_rank"] = candidate_summary["experiment_id"].map({
    "phase1a_003": 1,
    "phase1a_002": 2,
    "phase1a_007": 3,
    "phase1a_035": 4,
})

candidate_summary["candidate_note"] = candidate_summary["experiment_id"].map({
    "phase1a_003": "1순위: daytime-only, window 7, MLP current-day. 가장 안정적인 단순 후보.",
    "phase1a_002": "2순위: daytime-only, window 7, GRU. sequence DL 후보.",
    "phase1a_007": "3순위: daytime-only, window 14, MLP current-day. 성능은 좋지만 test participant 수가 적음.",
    "phase1a_035": "참고 후보: no-high-sleep-session, window 30, MLP current-day. sleep-session 제거 효과 비교용.",
})

candidate_summary = candidate_summary.sort_values("candidate_rank")

candidate_columns = [
    "candidate_rank",
    "experiment_id",
    "subset_name",
    "window",
    "model_family",
    "selected_threshold_from_validation_validation",
    "balanced_accuracy_validation",
    "roc_auc_validation",
    "average_precision_validation",
    "f1_validation",
    "balanced_accuracy_test",
    "roc_auc_test",
    "average_precision_test",
    "f1_test",
    "balanced_accuracy_gap",
    "participants",
    "total_samples",
    "mean_participant_accuracy",
    "min_participant_accuracy",
    "mean_participant_bal_acc",
    "min_participant_bal_acc",
    "participants_with_single_class_target",
    "candidate_note",
]

candidate_summary = candidate_summary[candidate_columns]

display(candidate_summary)

candidate_summary_path = OUTPUT_ROOT / "phase_1a_selected_candidate_summary.csv"
candidate_summary.to_csv(candidate_summary_path, index=False, encoding="utf-8-sig")

print("saved:", candidate_summary_path)

,candidate_rank,experiment_id,subset_name,window,model_family,selected_threshold_from_validation_validation,balanced_accuracy_validation,roc_auc_validation,average_precision_validation,f1_validation,...,f1_test,balanced_accuracy_gap,participants,total_samples,mean_participant_accuracy,min_participant_accuracy,mean_participant_bal_acc,min_participant_bal_acc,participants_with_single_class_target,candidate_note
1,1,phase1a_003,daytime_only,7,mlp_current_day,0.42,0.891944,0.908844,0.791541,0.862069,...,0.821549,0.047954,9,314,0.847957,0.600000,0.873183,0.777778,1,"1순위: daytime-only, window 7, MLP current-day. ..."
0,2,phase1a_002,daytime_only,7,gru,0.34,0.867635,0.891944,0.787546,0.832653,...,0.810289,0.035967,9,314,0.827793,0.600000,0.851131,0.777778,1,"2순위: daytime-only, window 7, GRU. sequence DL 후보."
2,3,phase1a_007,daytime_only,14,mlp_current_day,0.43,0.892363,0.906295,0.813508,0.846847,...,0.824074,0.063282,5,214,0.816325,0.666667,0.851849,0.803828,0,"3순위: daytime-only, window 14, MLP current-day...."
3,4,phase1a_035,no_high_sleep_session,30,mlp_current_day,0.51,0.928571,0.915751,0.721537,0.812500,...,0.829268,0.128461,4,106,0.760393,0.600000,0.814151,0.666667,0,"참고 후보: no-high-sleep-session, window 30, MLP c..."


saved: c:\workSpace\DeepLearnin_sleep\data\processed\deep_learning_subset_experiments\phase_1a_outputs\phase_1a_selected_candidate_summary.csv
